# RAG-Funktionen

Nun erweitern wir unseren Agenten um die Möglichkeit, Dokumente zu durchsuchen. 

Zunächst rufen wir einige Beispieldaten ab:

In [ ]:
%%bash
mkdir data
wget https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt -O data/paul_graham_essay.txt

Nun können wir ein Tool zur Dokumentensuche mit LlamaIndex erstellen. `VectorStoreIndex` verwendet standardmässig `text-embedding-ada-002` Embeddings von OpenAI, um den Text einzubetten und abzurufen.

Unsere modifizierte Version vom Basic Beispiel ist wie folgt:

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Global Settings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = Ollama(
    model="llama3.1:8b-instruct-q4_K_M",
    request_timeout=360.0,
    context_window=8000,
)

# Index aufbauen
documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()


def multiply(a: float, b: float) -> float:
    return a * b


async def search_documents(query: str) -> str:
    """
    Use this tool for ANY question that asks about the documents in ./data
    (e.g. 'author', 'college', 'essay', 'what did he do', etc.).
    Always use it before answering those questions.
    """
    resp = await query_engine.aquery(query)
    return str(resp)

agent = AgentWorkflow.from_tools_or_functions(
    [multiply, search_documents],
    llm=Settings.llm,
    system_prompt="""You are a helpful assistant that can perform calculations
    and search through documents to answer questions.""",
)


async def main():
    response = await agent.run(
        "What did the author do in college? Also, what's 7 * 8?"
    )
    return response


# Jupyter: direkt await verwenden
response = await main()
print(response)


Der Agent kann nun nahtlos zwischen der Nutzung des Taschenrechners und der Suche in Dokumenten zur Beantwortung von Fragen wechseln.

Quelle: [LlamaIndex](https://developers.llamaindex.ai/python/framework/getting_started/starter_example_local/#adding-rag-capabilities)